# mocsy examples
12/12 and 11/9/2015. [Emilio Mayorga](https://github.com/emiliom). Reproduce and extend Examples 1-3 from the mocsy Python documentation (`mainpage.txt`, or build it with `doxygen Doxyfile`).

Updated for Python 3 and the current build: build the module first with `make mocsy.so` (see `README.md`), and run this notebook from the `notebooks/` directory.

In [ ]:
import os, sys
# the compiled mocsy module is in the repository root (build it first, see README.md)
sys.path.insert(0, os.path.abspath(os.pardir))
import mocsy

## Example 1, Simple scalar variables
Functions return 1-dimensional numpy arrays. Scalar inputs return length-1 arrays.

For DATA input: DIC and ALk in mol/kg, in situ temperature, pressure.

In [ ]:
pH,pco2,fco2,co2,hco3,co3,OmegaA,OmegaC,BetaD,DENis,p,Tis = \
    mocsy.mvars.vars(temp=18, sal=35, alk=2300.e-6, dic=2000.e-6, sil=0, phos=0, 
                patm=1, depth=100, lat=0, 
                optcon='mol/kg', optt='Tinsitu', optp='db', 
                optb="u74", optk1k2='l', optkf="dg", optgas='Pinsitu')
print(pH,pco2,fco2,co2,hco3,co3,OmegaA,OmegaC,BetaD,DENis,p,Tis)

Compute the CONSTANTS with the same scalar input for S, T, and P (as above) but change to the newer options published since the best-practices guide: Lee et al. (2010) for total boron and Millero (2010) for K<sub>1</sub> and K<sub>2</sub>:

In [ ]:
Kh,K1,K2,Kb,Kw,Ks,Kf,Kspc,Kspa,K1p,K2p,K3p,Ksi,St,Ft,Bt = \
    mocsy.mconstants.constants(temp=18, sal=35, patm=1, depth=0,lat=0, 
                     optt='Tinsitu', optp='m', 
                     optb="l10", optk1k2='m10', optkf="dg", optgas='Ppot')
print(Kh,K1,K2,Kb,Kw,Ks,Kf,Kspc,Kspa,K1p,K2p,K3p,Ksi,St,Ft,Bt)

For MODEL input: DIC and Alk in mol/m3, potential temperature, and depth. Also do NOT specify last 4 options, thus using defaults (optb='l10', optk1k2='l', optkf='pf', optgas='Pinsitu').

In [ ]:
pH,pco2,fco2,co2,hco3,co3,OmegaA,OmegaC,BetaD,DENis,p,Tis = \
    mocsy.mvars.vars(temp=18, sal=35, alk=2300*1028e-6, dic=2000*1028e-6, 
                sil=0, phos=0, patm=1, depth=100, lat=0, 
                optcon='mol/m3', optt='Tpot', optp='m')
print(pH,pco2,fco2,co2,hco3,co3,OmegaA,OmegaC,BetaD,DENis,p,Tis)

## Example 2,  Simple arrays (numpy)
Also demonstrate import into Pandas DataFrame and plotting.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
# Make dummy array with 11 members
one = np.ones(11, dtype='float32')

sal   = 35*one
temp  = 2*one
patm  = 1*one
depth = np.arange(0, 11000, 1000, dtype='float32')  # units in 'm'

In [ ]:
# Compute in situ density (3 steps using 2 mocsy routines):
# a) Specify latitude = 60°S for depth to pressure conversion formula
lat = -60*one
# b) Compute pressure (db) from depth (m) and latitude following Saunders (1981)
pfd = mocsy.mdepth2press.depth2press(depth,lat)  # units in 'db' 
# c) Compute in situ density (kg/m3) from salinity (psu), in-situ temp (C), and pressure (db)
rhois = mocsy.mrhoinsitu.rhoinsitu(sal, temp, pfd)

In [ ]:
# Specify surface concentrations typical of 60°S (from GLODAP and WOA2009)
alk   = 2295*one # umol/kg
dic   = 2154*one # umol/kg
sio2  = 50.*one  # umol/L
po4   = 1.8*one  # umol/L

In [ ]:
# Convert nutrient units(mol/L) to model units (mol/m3)
sio2 = sio2 * 1.0e-3   
po4  = po4  * 1.0e-3 
# Convert Alk and DIC units (umol/kg) to model units (mol/m3)
dic = dic * rhois * 1e-6
alk = alk * rhois * 1e-6

In [ ]:
# Compute carbonate system variables, using 'model' options.
# Note that original example code had an error: 
# patm argument was passed as scalar rather than array shaped like one
response_tup = mocsy.mvars.vars(temp=temp, sal=sal, alk=alk, dic=dic, 
                           sil=sio2, phos=po4, patm=patm, depth=depth, lat=lat, 
                           optcon='mol/m3', optt='Tpot', optp='m')
pH,pco2,fco2,co2,hco3,co3,OmegaA,OmegaC,BetaD,DENis,p,Tis = response_tup

In [ ]:
depth, pH, OmegaA

### Import into Pandas DataFrame

In [ ]:
data_for_pd = dict(depth=depth, 
                   temp=temp, sal=sal, alk=alk, dic=dic, sil=sio2, phos=po4,
                   pH=pH, pCO2=pco2, fCO2=fco2, CO3=co3, OmegaA=OmegaA, OmegaC=OmegaC
                  )

In [ ]:
data_df = pd.DataFrame(data_for_pd, 
                       columns=['depth', 'temp', 'sal', 'alk', 'dic', 'sil', 'phos',
                                'pH', 'pCO2', 'fCO2', 'CO3', 'OmegaA', 'OmegaC']
                      )
data_df.set_index('depth', drop=False, inplace=True)

In [ ]:
data_df

### Create simple, easily generated figure using  Pandas plot

In [ ]:
data_df['OmegaA'].plot(title='Omega Aragonite (OmegaA) vs depth');

### Create a customized property vs depth plot showing two variables using matplotlib

In [ ]:
fig, ax1 = plt.subplots(figsize=(5,7))

ax1.plot(data_df['OmegaA'], data_df['depth'], lw=2, color="blue")
ax1.set_xlabel(r"$\Omega_A (fraction)$", fontsize=16, color="blue")
for label in ax1.get_xticklabels():
    label.set_color("blue")
    
ax2 = ax1.twiny()
ax2.plot(data_df['pCO2'], data_df['depth'], lw=2, color="red")
ax2.set_xlabel(r"$pCO_2 (\mu atm)$", fontsize=16, color="red")
for label in ax2.get_xticklabels():
    label.set_color("red")

ax1.invert_yaxis()
ax1.set_ylabel(r"$depth (m)$", fontsize=16);

Note that pCO2 here is calculated for "in-situ" conditions, a capability that apparently is fairly unique to mocsy. See    
[Orr, J. C. and Epitalon, J.-M.: Improved routines to model the ocean carbonate system: mocsy 2.0, Geosci. Model Dev., 8, 485-499, doi:10.5194/gmd-8-485-2015, 2015](https://doi.org/10.5194/gmd-8-485-2015)

## Example 3, Read input from a .csv (spreadsheet) file
Also demonstrate import into Pandas DataFrame and plotting.

In [ ]:
# Read input .csv file shipped with this repository
infile = os.path.join(os.pardir, "examples", "data", "DIC-Alk-P_vary_input.csv")

In [ ]:
# Read .csv input file with 7 columns: flag, Alk(mol/kg), DIC(mol/kg), salinity(psu), 
#   temp(C), press(db), PO43-(mol/kg), SiO2(mol/kg)
indata = np.loadtxt(infile, delimiter=',', dtype='float32', skiprows=1)

In [ ]:
n = len(indata)
alk = indata[:,1]
dic = indata[:,2]
sal = indata[:,3]
temp  = indata[:,4]
depth = indata[:,5] * 10 #Convert from bars to decibars
phos  = indata[:,6]
sil   = indata[:,7]

# Specify that latitude = 45°N for all samples (for depth to pressure conversion)
# note: "lat" is required input for mocsy, but results are only weakly sensitive
lat = np.full_like(depth, 45)
# Specify the atmospheric pressure for all samples (1 atm)
patm = np.full_like(depth, 1)

In [ ]:
# Recover computed carbonate system variables
response_tup = mocsy.mvars.vars(temp=temp, sal=sal, alk=alk, dic=dic, 
                           sil=sil, phos=phos, patm=patm, depth=depth, lat=lat, 
                           optcon='mol/kg', optt='Tinsitu', optp='db', 
                           optb="u74", optk1k2='l', optkf="dg", optgas='Pzero')
pH, pco2, fco2, co2, hco3, co3, OmegaA, OmegaC, BetaD, rhoSW, p, tempis = response_tup

**Notes for the above calculation:**
1. Original example code had an error: patm argument was missing
2. Changed the original code to use `optgas='Pzero'`, to expose the `optgas` option and match the pre-existing output on `examples/data/DIC-Alk-P-vary-from-mocsy.csv`. From the mvars module in the Fortran code (`src/vars.f90`), regarding `Pzero`: *"'zero order' fCO2 and pCO2 (typical approach, which is flawed) considers in situ T & only atm pressure (hydrostatic=0)"*

### Import into Pandas DataFrame

In [ ]:
data_for_pd = dict(depth=depth, 
                   temp=temp, sal=sal, alk=alk, dic=dic, sil=sil, phos=phos,
                   pH=pH, pCO2=pco2, fCO2=fco2, CO3=co3, OmegaA=OmegaA, OmegaC=OmegaC
                  )

data_df = pd.DataFrame(data_for_pd, 
                       columns=['depth', 'temp', 'sal', 'alk', 'dic', 'sil', 'phos',
                                'pH', 'pCO2', 'fCO2', 'CO3', 'OmegaA', 'OmegaC']
                      )
data_df.set_index('depth', drop=False, inplace=True)
data_df

This output is a good match to the output csv file in this repository, `examples/data/DIC-Alk-P-vary-from-mocsy.csv`.

In [ ]:
fig, ax1 = plt.subplots(figsize=(5,7))

ax1.plot(data_df['OmegaA'], data_df['depth'], lw=2, color="blue")
ax1.set_xlabel(r"$\Omega_A (fraction)$", fontsize=16, color="blue")
for label in ax1.get_xticklabels():
    label.set_color("blue")
    
ax2 = ax1.twiny()
ax2.plot(data_df['pH'], data_df['depth'], lw=2, color="red")
ax2.set_xlabel(r"$pH (total scale)$", fontsize=16, color="red")
for label in ax2.get_xticklabels():
    label.set_color("red")

ax1.invert_yaxis()
ax1.set_ylabel(r"$depth (m)$", fontsize=16);